# EXPLAIN and EXPLAIN ANALYZE

## Overview
`EXPLAIN` shows the **query plan** -- the strategy the database engine will use to execute a query -- without running it. `EXPLAIN ANALYZE` actually executes the query and reports both the estimated and actual costs.

**PostgreSQL EXPLAIN ANALYZE output:**
```
Seq Scan on observations  (cost=0.00..1.25 rows=32 width=68)
                           (actual time=0.012..0.018 rows=32 loops=1)
  Filter: (site_id = 1)
  Rows Removed by Filter: 28
Planning Time: 0.1 ms
Execution Time: 0.2 ms
```

**Key fields to read:**

| Field | Meaning |
|---|---|
| `cost=0.00..1.25` | Estimated cost: startup..total (in planner cost units, not ms) |
| `rows=32` | Estimated row count |
| `actual time=0.012..0.018` | Real time in ms: first row..last row |
| `actual rows=4` | Real row count returned |
| `loops=N` | How many times this node was executed (e.g. nested loop inner side) |
| `width=68` | Estimated average row width in bytes |

**Common plan node types:**

| Node | Meaning |
|---|---|
| `Seq Scan` | Full table scan -- reads every row |
| `Index Scan` | Uses an index; fetches heap rows for each match |
| `Index Only Scan` | All needed columns are in the index; no heap access |
| `Bitmap Heap Scan` | Collects index matches into a bitmap, then fetches heap pages |
| `Hash Join` | Builds a hash table on the smaller input; probes with the larger |
| `Nested Loop` | For each row in outer, scans inner (good when inner is small/indexed) |
| `Merge Join` | Both inputs sorted; merges in order (good for large sorted inputs) |
| `Sort` | Sort step -- often appears before Merge Join or when ORDER BY lacks index |

**SQLite note:** SQLite uses `EXPLAIN QUERY PLAN` (no ANALYZE variant). It shows scan type and whether an index is used, but does not report actual row counts or timing. All PostgreSQL EXPLAIN output in this notebook is shown as annotated examples.

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE sites (site_id INTEGER PRIMARY KEY, site_name TEXT NOT NULL, region TEXT, latitude REAL, longitude REAL, elevation_m REAL, established_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE species (species_id INTEGER PRIMARY KEY, common_name TEXT NOT NULL, scientific_name TEXT NOT NULL UNIQUE, taxon_group TEXT, native INTEGER DEFAULT 1, at_risk INTEGER DEFAULT 0);CREATE TABLE field_crew (crew_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, role TEXT, certified INTEGER DEFAULT 1);CREATE TABLE observations (obs_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), species_id INTEGER REFERENCES species(species_id), crew_id INTEGER REFERENCES field_crew(crew_id), obs_date TEXT NOT NULL, count_individuals INTEGER, life_stage TEXT, method TEXT, notes TEXT);CREATE TABLE water_quality (wq_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), crew_id INTEGER REFERENCES field_crew(crew_id), sample_date TEXT NOT NULL, ph REAL, dissolved_o2 REAL, turbidity_ntu REAL, temp_c REAL, conductivity_us REAL);INSERT INTO sites VALUES (1,'Fundy Estuary','Atlantic',45.78,-64.52,3.2,'2018-04-01',1),(2,'Kejimkujik Lake','Atlantic',44.43,-65.21,156.0,'2017-06-15',1),(3,'Presqu ile Point','Great Lakes',43.99,-77.72,75.5,'2019-03-20',1),(4,'Rondeau Bay','Great Lakes',42.31,-81.87,176.0,'2018-09-10',1),(5,'Athabasca Delta','Boreal',58.72,-110.87,213.0,'2016-01-15',1),(6,'Wapusk Tundra','Boreal',57.92,-93.15,45.0,'2017-07-01',1),(7,'Clayoquot Sound','Pacific',49.15,-125.93,12.0,'2015-05-20',1),(8,'Boundary Bay','Pacific',49.01,-122.97,1.5,'2016-08-01',1),(9,'Lac Saint-Pierre','St Lawrence',46.19,-72.87,8.0,'2018-02-14',1),(10,'Baie des Chaleurs','Atlantic',48.01,-65.72,5.0,'2019-11-01',0);INSERT INTO species VALUES (1,'Atlantic Salmon','Salmo salar','Fish',1,1),(2,'Great Blue Heron','Ardea herodias','Bird',1,0),(3,'Wood Duck','Aix sponsa','Bird',1,0),(4,'Spotted Turtle','Clemmys guttata','Reptile',1,1),(5,'Common Loon','Gavia immer','Bird',1,0),(6,'Muskrat','Ondatra zibethicus','Mammal',1,0),(7,'Northern Pike','Esox lucius','Fish',1,0),(8,'Bald Eagle','Haliaeetus leucocephalus','Bird',1,0),(9,'American Bittern','Botaurus lentiginosus','Bird',1,1),(10,'Snapping Turtle','Chelydra serpentina','Reptile',1,0),(11,'Rainbow Trout','Oncorhynchus mykiss','Fish',0,0),(12,'Ring-necked Duck','Aythya collaris','Bird',1,0),(13,'Beaver','Castor canadensis','Mammal',1,0),(14,'River Otter','Lontra canadensis','Mammal',1,0),(15,'Painted Turtle','Chrysemys picta','Reptile',1,0);INSERT INTO field_crew VALUES (1,'Dr. Aroha Ngata','Biologist',1),(2,'Liam Chen','Technician',1),(3,'Fatima Al-Rashid','Biologist',1),(4,'James MacLeod','Technician',1),(5,'Sofia Petrov','Student',0);INSERT INTO observations VALUES (1,1,1,1,'2023-04-10',12,'Adult','Electrofishing',NULL),(2,1,2,2,'2023-04-10',3,'Adult','Visual',NULL),(3,2,5,1,'2023-04-15',2,'Adult','Auditory',NULL),(4,2,4,3,'2023-04-15',8,'Juvenile','Visual',NULL),(5,3,2,2,'2023-05-01',5,'Adult','Visual',NULL),(6,3,3,4,'2023-05-01',7,'Adult','Visual',NULL),(7,4,10,3,'2023-05-10',2,'Adult','Visual',NULL),(8,4,7,1,'2023-05-10',4,'Adult','Electrofishing',NULL),(9,5,13,2,'2023-05-20',6,'Adult','Camera Trap',NULL),(10,5,14,3,'2023-05-20',1,'Adult','Visual',NULL),(11,6,8,1,'2023-06-01',3,'Adult','Visual',NULL),(12,6,6,4,'2023-06-01',9,'Adult','Trap',NULL),(13,7,14,3,'2023-06-15',2,'Adult','Visual',NULL),(14,7,5,2,'2023-06-15',4,'Adult','Auditory',NULL),(15,8,2,1,'2023-07-01',12,'Adult','Visual',NULL),(16,8,8,4,'2023-07-01',2,'Adult','Visual',NULL),(17,9,1,3,'2023-07-10',7,'Adult','Electrofishing',NULL),(18,9,9,1,'2023-07-10',1,'Adult','Visual','First sighting this season'),(19,1,4,2,'2023-08-05',1,'Adult','Visual',NULL),(20,2,13,4,'2023-08-05',4,'Adult','Camera Trap',NULL),(21,3,6,3,'2023-08-12',11,'Adult','Trap',NULL),(22,4,2,2,'2023-08-12',7,'Adult','Visual',NULL),(23,5,8,1,'2023-09-01',1,'Adult','Visual',NULL),(24,6,14,3,'2023-09-01',3,'Adult','Visual',NULL),(25,7,1,4,'2023-09-15',18,'Adult','Electrofishing',NULL),(26,8,5,2,'2023-09-15',6,'Adult','Visual',NULL),(27,9,4,3,'2023-09-20',2,'Juvenile','Visual',NULL),(28,1,7,1,'2023-10-01',3,'Adult','Electrofishing',NULL),(29,2,1,2,'2023-10-01',9,'Adult','Electrofishing',NULL),(30,3,8,4,'2023-10-10',4,'Adult','Visual',NULL),(31,4,5,1,'2023-10-15',3,'Adult','Auditory',NULL),(32,5,2,3,'2023-10-15',8,'Adult','Visual',NULL);INSERT INTO water_quality VALUES (1,1,1,'2023-04-10',7.2,9.1,3.4,8.5,142.0),(2,1,2,'2023-06-15',7.4,8.6,4.2,14.2,138.0),(3,1,3,'2023-08-20',7.1,7.8,5.1,19.6,145.0),(4,2,1,'2023-04-15',6.8,10.2,1.2,7.1,98.0),(5,2,4,'2023-07-01',6.9,9.4,1.8,16.3,102.0),(6,3,2,'2023-05-01',7.6,9.8,2.3,11.4,188.0),(7,3,3,'2023-08-01',7.5,8.2,3.7,20.1,192.0),(8,4,1,'2023-05-10',7.8,9.5,1.9,13.0,205.0),(9,4,4,'2023-09-05',7.7,8.9,2.5,18.4,198.0),(10,5,2,'2023-05-20',7.3,10.8,0.8,9.2,55.0),(11,5,1,'2023-09-10',7.2,9.6,1.1,15.7,58.0),(12,6,3,'2023-06-01',6.5,11.2,0.5,6.8,32.0),(13,7,2,'2023-06-15',7.9,9.0,1.4,12.5,310.0),(14,8,4,'2023-07-01',7.8,8.7,2.1,17.2,295.0),(15,9,1,'2023-07-10',7.6,9.3,2.8,14.8,178.0);""")
conn.commit()
print("Ecological schema ready.")
for t in ["sites","species","field_crew","observations","water_quality"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

Ecological schema ready.
  sites: 10 rows
  species: 15 rows
  field_crew: 5 rows
  observations: 32 rows
  water_quality: 15 rows


---
## SQLite EXPLAIN QUERY PLAN

In [2]:
print("=== SQLite EXPLAIN QUERY PLAN: no index vs with index ===")
print()

# No index on obs_date
plan = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT * FROM observations WHERE obs_date = '2023-07-10'
""").fetchall()
print("Query: WHERE obs_date = '2023-07-10' (no index)")
for row in plan:
    print(" ", row)

conn.execute("CREATE INDEX idx_obs_date ON observations (obs_date)")
conn.commit()

plan2 = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT * FROM observations WHERE obs_date = '2023-07-10'
""").fetchall()
print()
print("After CREATE INDEX idx_obs_date:")
for row in plan2:
    print(" ", row)

# Join plan
plan3 = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT o.obs_id, s.site_name, sp.common_name
    FROM   observations o
    JOIN   sites    s  ON o.site_id    = s.site_id
    JOIN   species  sp ON o.species_id = sp.species_id
    WHERE  o.obs_date > '2023-07-01'
""").fetchall()
print()
print("Join plan (observations + sites + species):")
for row in plan3:
    print(" ", row)


=== SQLite EXPLAIN QUERY PLAN: no index vs with index ===

Query: WHERE obs_date = '2023-07-10' (no index)
  (2, 0, 0, 'SCAN observations')

After CREATE INDEX idx_obs_date:
  (3, 0, 0, 'SEARCH observations USING INDEX idx_obs_date (obs_date=?)')

Join plan (observations + sites + species):
  (5, 0, 0, 'SEARCH o USING INDEX idx_obs_date (obs_date>?)')
  (9, 0, 0, 'SEARCH s USING INTEGER PRIMARY KEY (rowid=?)')
  (12, 0, 0, 'SEARCH sp USING INTEGER PRIMARY KEY (rowid=?)')


---
## Reading PostgreSQL EXPLAIN ANALYZE output

In [3]:
print("=== Annotated PostgreSQL EXPLAIN ANALYZE examples ===")
print()

examples = [
    ("Seq Scan -- no index, small table",
     """Seq Scan on observations  (cost=0.00..1.60 rows=32 width=72)
                               (actual time=0.009..0.025 rows=32 loops=1)
Planning Time: 0.08 ms
Execution Time: 0.04 ms""",
     "Reading every row in the table. Efficient for small tables or when most rows qualify."),

    ("Index Scan -- selective filter, index on obs_date",
     """Index Scan using idx_obs_date on observations
                               (cost=0.14..8.16 rows=2 width=72)
                               (actual time=0.031..0.033 rows=2 loops=1)
  Index Cond: (obs_date = '2023-07-10')
Planning Time: 0.12 ms
Execution Time: 0.05 ms""",
     "Used the index to jump directly to matching rows. Fast because only 2 rows qualify."),

    ("Hash Join -- joining observations to sites",
     """Hash Join  (cost=1.23..2.89 rows=32 width=48)
                (actual time=0.052..0.089 rows=32 loops=1)
  Hash Cond: (observations.site_id = sites.site_id)
  ->  Seq Scan on observations  (cost=0.00..1.60 rows=32 width=28)
  ->  Hash  (cost=1.10..1.10 rows=10 width=24)
        Buckets: 1024  Batches: 1  Memory Usage: 9kB
        ->  Seq Scan on sites  (cost=0.00..1.10 rows=10 width=24)
Planning Time: 0.18 ms
Execution Time: 0.11 ms""",
     "Built a hash table on sites (smaller), probed with observations (larger). Good when one side is small."),

    ("Sort + row estimate mismatch -- warning sign",
     """Sort  (cost=3.42..3.50 rows=32 width=72)
          (actual time=0.122..0.125 rows=32 loops=1)
  Sort Key: obs_date
  Sort Method: quicksort  Memory: 29kB
  ->  Seq Scan on observations  (cost=0.00..1.60 rows=32 width=72)
                                (actual time=0.011..0.022 rows=32 loops=1)
Planning Time: 0.09 ms
Execution Time: 0.15 ms

-- Estimated rows=32, actual rows=32: good estimate.
-- If estimated rows=3, actual rows=3200: stale statistics.
-- Fix: ANALYZE observations;  or  VACUUM ANALYZE observations;""",
     "Sort node appears when ORDER BY cannot use an index. Large sorts spill to disk (Sort Method: external merge)."),
]

for title, plan, note in examples:
    print(f"--- {title} ---")
    print(plan)
    print(f"Note: {note}")
    print()


=== Annotated PostgreSQL EXPLAIN ANALYZE examples ===

--- Seq Scan -- no index, small table ---
Seq Scan on observations  (cost=0.00..1.60 rows=32 width=72)
                               (actual time=0.009..0.025 rows=32 loops=1)
Planning Time: 0.08 ms
Execution Time: 0.04 ms
Note: Reading every row in the table. Efficient for small tables or when most rows qualify.

--- Index Scan -- selective filter, index on obs_date ---
Index Scan using idx_obs_date on observations
                               (cost=0.14..8.16 rows=2 width=72)
                               (actual time=0.031..0.033 rows=2 loops=1)
  Index Cond: (obs_date = '2023-07-10')
Planning Time: 0.12 ms
Execution Time: 0.05 ms
Note: Used the index to jump directly to matching rows. Fast because only 2 rows qualify.

--- Hash Join -- joining observations to sites ---
Hash Join  (cost=1.23..2.89 rows=32 width=48)
                (actual time=0.052..0.089 rows=32 loops=1)
  Hash Cond: (observations.site_id = sites.site_id)


---
## Key EXPLAIN patterns and what to look for

In [4]:
print("=== EXPLAIN diagnostic checklist ===")
checks = [
    ("Seq Scan on large table",
     "Consider adding an index if a small % of rows qualify (< ~5-10%)"),
    ("rows estimate >> actual rows",
     "Run ANALYZE to update statistics; check for data skew"),
    ("Sort (external merge)",
     "Sort spilled to disk; increase work_mem or add index to avoid sort"),
    ("Nested Loop with large outer",
     "Each outer row triggers a full scan of inner; ensure inner is indexed"),
    ("Hash Join with Batches > 1",
     "Hash table spilled to disk; increase work_mem"),
    ("Bitmap Heap Scan + Recheck Cond",
     "Normal pattern for range queries; Recheck means lossy bitmap was used"),
    ("Index Scan but actual rows >> estimated",
     "Stale statistics; ANALYZE the table"),
    ("loops=N on inner of Nested Loop",
     "N = number of times inner side was executed; high N with Seq Scan = bad"),
]
print(f"{'Plan signal':<40s}  {'What to do'}")
print("-" * 80)
for signal, action in checks:
    print(f"  {signal:<38s}  {action}")

print()
print("=== Useful PostgreSQL EXPLAIN options ===")
print("""
  EXPLAIN (ANALYZE, BUFFERS) query;
      -- Shows cache hit/miss rates (shared_blks_hit vs shared_blks_read)

  EXPLAIN (ANALYZE, FORMAT JSON) query;
      -- Machine-readable output; paste into explain.depesz.com or explain.dalibo.com

  EXPLAIN (ANALYZE, TIMING OFF) query;
      -- Reduces timing overhead for very fast queries; more accurate row counts

  SET enable_seqscan = OFF;
  EXPLAIN query;
      -- Forces the planner away from Seq Scans; useful to see what index plan would look like
  SET enable_seqscan = ON;  -- always reset after
""")
conn.close()


=== EXPLAIN diagnostic checklist ===
Plan signal                               What to do
--------------------------------------------------------------------------------
  Seq Scan on large table                 Consider adding an index if a small % of rows qualify (< ~5-10%)
  rows estimate >> actual rows            Run ANALYZE to update statistics; check for data skew
  Sort (external merge)                   Sort spilled to disk; increase work_mem or add index to avoid sort
  Nested Loop with large outer            Each outer row triggers a full scan of inner; ensure inner is indexed
  Hash Join with Batches > 1              Hash table spilled to disk; increase work_mem
  Bitmap Heap Scan + Recheck Cond         Normal pattern for range queries; Recheck means lossy bitmap was used
  Index Scan but actual rows >> estimated  Stale statistics; ANALYZE the table
  loops=N on inner of Nested Loop         N = number of times inner side was executed; high N with Seq Scan = bad

=== Useful 

---
## Common Pitfalls

**1. Using EXPLAIN without ANALYZE and trusting estimated rows**
`EXPLAIN` (without ANALYZE) shows estimates based on table statistics. A table last analysed 6 months ago may have wildly wrong row estimates. Always use `EXPLAIN ANALYZE` for diagnosis; use plain `EXPLAIN` only when you cannot afford to run the query.

**2. Not resetting session-level planner settings after testing**
`SET enable_seqscan = OFF` disables sequential scans for the rest of the session. Forgetting to reset it with `SET enable_seqscan = ON` means all subsequent queries in that session avoid seq scans -- potentially choosing slower plans. Always reset after testing.

**3. Misreading cost units as milliseconds**
`cost=0.00..245.32` is in planner cost units (roughly proportional to disk page reads), not milliseconds. A cost of 245 on a fast SSD might execute in 1 ms. Compare costs between plan alternatives for the same query, not across different hardware or queries.

**4. Ignoring the loops field in Nested Loop nodes**
`(actual time=0.5..1.2 rows=5 loops=200)` means the inner side ran 200 times. Total time for that node is `1.2 ms * 200 = 240 ms`. The per-loop time looks small but the total is large. Always multiply `actual time` by `loops` to get the real contribution.

**5. Running EXPLAIN ANALYZE on mutating queries in production**
`EXPLAIN ANALYZE DELETE FROM ...` actually deletes the rows while measuring. Wrap in a transaction and ROLLBACK, or use a test environment: `BEGIN; EXPLAIN ANALYZE DELETE ...; ROLLBACK;`

**6. Trusting EXPLAIN on very small tables**
On a 32-row table, the planner almost always chooses Seq Scan because the overhead of an index lookup exceeds the cost of reading 32 rows. This is correct behaviour -- it does not mean your indexes are broken. Add enough data to test realistic plan decisions (thousands of rows minimum).


---
*sql_methods_library - Samantha McGarrigle*